# 28. Domain Lexicon v1 — 초기 항목

| | |
|---|---|
| 만드는 것 | `data/scent_knowledge/domain_lexicon_v1.csv` |
| 근거 | `spec.md` §4.3 (스키마·우선순위 1~6, 매핑 규칙) · `DECISIONS.md` N1 · N2 |
| API | **호출하지 않는다** |
| 작성 | 2026-09-10 |

## 이 사전이 담는 것과 담지 않는 것

`spec.md` §4.3: *"**표현을 나열하지 않는다.** 사전 커버리지를 재보니 새 쿼리의 8.2%였다.
나열로는 덮을 수 없고, 덮을 필요도 없다. **LLM이 못 하는 지점만 담으면 된다.**"*

따라서 `우디 → woody` 처럼 LLM이 잘 처리하는 외래어 파생은 담지 않는다
(노트북 26에서 외래어 파생의 1순위 accord 일치가 100%였다).
그 표현들은 채점기용 별칭표 `25_stage1_scoring_alias_v1.csv` 40행에 이미 있고,
**이 사전은 그것을 복제하지 않는다.**

담는 것은 §4.3의 우선순위 1~6이다.

| 순위 | 담을 것 | 이 노트북에서 |
|---|---|---|
| 1 | 향수 전문 용어 | **담는다** — `candidate_type=FIELD` |
| 2 | 코퍼스 지지도 | **모든 행의 `corpus_support`로 담는다** |
| 3 | 한국어 드리프트 (머스크) | **담는다** |
| 4 | 문맥 분기 (깨끗한) | **담는다** |
| 5 | 매핑 금지 | **담는다** — `NO_MAPPING` |
| 6 | 방언 `달달` | **담는다** |
| 7 | 은어·비유 | 담지 않는다 (`DECISIONS.md` N1 — 후순위) |

## 0. 실행 조건과 한계

### 스키마 변경 1건 — `candidate_type`에 `FIELD` 추가

`spec.md` §4.3의 `candidate_type`은 `ACCORD` / `NOTE` 뿐이었다. 그런데 우선순위 **1번**
(향수 전문 용어, 실사용 18.1%)은 `탑노트 → notes.tiered.top`, `지속력 → longevity` 처럼
**Fragrantica 스키마 필드**를 가리키므로 담을 칸이 없었다.

`FIELD` 값을 추가한다. 사용자 승인 2026-09-10.

### 근거 등급을 매핑의 출처로 정의한다

`spec.md` §4.3의 세 등급을 이 노트북에서 다음과 같이 적용한다.

| 등급 | 이 노트북에서의 뜻 | 예 |
|---|---|---|
| `VERIFIED` | 스키마 필드 · accord/note 이름 동일성 · 코퍼스 측정값이 매핑을 결정 | `탑노트 → notes.tiered.top` |
| `TEAM` | 팀 문서 `perfume_14families_korean_descriptors.md`에 **출처 행이 있다** | `빨래 → soapy` (청결감 축) |
| `LLM` | 출처 없이 AI가 초안으로 고름. **사람 검토 후 `TEAM`으로 승격** | `달달 → caramel` |

`AGENTS.md`: *"Do not invent mappings such as an abstract phrase to scent features
without a defined evidence or modeling method."* → 위 세 방법 밖의 매핑은 만들지 않고,
근거가 없으면 `NO_MAPPING`으로 둔다. 팀 문서 자신이 `무난한`에 대해
*"(매핑 근거 미확보)"* 라고 적어둔 선례를 따른다.

### 한계 — 결과를 읽을 때 반드시 함께 볼 것

- **`TEAM` 등급은 "팀 문서에 표현이 있다"까지만 보증한다.** 팀 문서는 표현을 **향 계열**로
  정리한 것이고 accord를 지정하지 않았다. 계열 → accord 단계는 이 노트북의 판단이다.
  문서 자신도 경고한다 — *"`포근한`, `깨끗한` 같은 단어는 패밀리를 단독 판별하는 단어가
  아니라 방향을 잡는 단어다."*
- **의미 판정을 받지 않았다.** 검색이 되는지는 계산했지만(§4) `soapy+fresh`가
  `빨래`의 *옳은* 번역인지는 사람이 판정할 문제다. 모든 매핑 행은 `status=candidate`다.
- **질감층(`무거운`·`가벼운`·`강한`·`묵직한`)을 넣지 않았다.** `spec.md` §8 남은 작업 5번이
  미결정이다. `25_stage1_scoring_alias_v1.csv`의 `heavy`/`light`/`strong`은 채점 전용 키이며
  92개 accord에도 note에도 없다.
- **`잔향`은 두 뜻이 겹친다.** 지속력인지 베이스노트인지 문맥에 따라 갈리므로 두 행으로 뒀다.
- **쿼리 12~200개 규모의 근거다.** 실사용 설문 155건 중 143건은 아직 라벨링되지 않았다.

### 하지 않는 것

평가 데이터 수정, 기존 노트북 수정·실행, `25_stage1_scoring_alias_v1.csv` 복제·수정,
LLM 호출, 은어·비유 항목 추가, 질감층 매핑.

In [1]:
import hashlib
import json
import pathlib

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 46)
pd.set_option("display.width", 240)

# True면 계산과 표시만 하고 파일을 만들지 않는다.
REPORT_ONLY = False

LEXICON_VERSION = "v1"
print("REPORT_ONLY:", REPORT_ONLY, "/ LEXICON_VERSION:", LEXICON_VERSION)

REPORT_ONLY: False / LEXICON_VERSION: v1


## 1. 경로 · 입력 해싱 · 쓰기 가드

25·26·27번과 같은 방식이다. 다만 **쓰기 허용 위치가 두 곳**이다.

`AGENTS.md`의 역할 구분을 따른다 — 사전은 나중 코드가 재사용하는 데이터이므로 `data/`,
빌드 리포트는 실험 기록이므로 `analysis_outputs/`.

In [2]:
PROJECT_ROOT = pathlib.Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"
KNOWLEDGE_DIR = PROJECT_ROOT / "data" / "scent_knowledge"

INPUT_PATHS = {
    "accord_dictionary": OUTPUT_DIR / "10_accord_dictionary.csv",
    "note_vocabulary": OUTPUT_DIR / "17_fragrantica_note_vocabulary.csv",
    "scoring_alias": OUTPUT_DIR / "25_stage1_scoring_alias_v1.csv",
    "korean_lexicon": KNOWLEDGE_DIR / "korean_scent_lexicon_v0_1.csv",
    "team_doc": KNOWLEDGE_DIR / "source" / "perfume_14families_korean_descriptors.md",
    "perfumes": PROJECT_ROOT / "perfumes.csv",
}
OUTPUT_PATHS = {
    "lexicon": KNOWLEDGE_DIR / f"domain_lexicon_{LEXICON_VERSION}.csv",
    "report": OUTPUT_DIR / "28_domain_lexicon_build_report.md",
}


def sha256_file(path):
    """파일의 SHA-256 hex digest. str."""
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


missing = [str(p) for p in INPUT_PATHS.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")
input_hashes_before = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}

FROZEN_FILES = {p.resolve() for p in INPUT_PATHS.values()}
PROTECTED_EXTRA = {
    (PROJECT_ROOT / "evaluation_data" / "stage1"
     / "13_stage1_golden_set_v1_200.xlsx").resolve(),
    (PROJECT_ROOT / "evaluation_data" / "stage1"
     / "16_golden_set_quality_audit_reviewed.csv").resolve(),
    (PROJECT_ROOT / "evaluation_data" / "semantic_bridge"
     / "22_pilot_human_evaluation.csv").resolve(),
    (OUTPUT_DIR / "16_golden_set_quality_audit_candidates.csv").resolve(),
}
# 쓰기가 허용된 정확한 경로 2개. 그 밖은 전부 거부한다.
ALLOWED_WRITES = {p.resolve() for p in OUTPUT_PATHS.values()}


def write_output(path, writer):
    """OUTPUT_PATHS 의 두 경로에만 쓴다. REPORT_ONLY면 생략. pathlib.Path 또는 None."""
    path = pathlib.Path(path).resolve()
    if path not in ALLOWED_WRITES:
        raise RuntimeError(f"쓰기 허용 경로가 아닙니다: {path}")
    if path in FROZEN_FILES or path in PROTECTED_EXTRA:
        raise RuntimeError(f"보호 파일 덮어쓰기 시도: {path.name}")
    if REPORT_ONLY:
        print(f"[REPORT_ONLY] 저장 생략: {path.name}")
        return None
    writer(path)
    print(f"저장: {path.relative_to(PROJECT_ROOT)}")
    return path


display(pd.Series(input_hashes_before, name="sha256").str.slice(0, 16).to_frame())

,sha256
accord_dictionary,567e91370e575731
note_vocabulary,3b3c340c4da9eab9
scoring_alias,7a3b852d356d016f
korean_lexicon,49250e9a3e5c98d4
team_doc,356762eb194bd8d3
perfumes,cec1ea0b49885303


## 2. 사전 등록 — 항목을 쓰기 전에 규칙을 고정한다

`spec.md` §4.3의 매핑 규칙과 §0의 근거 방법을 코드가 검사할 수 있는 형태로 적는다.
항목 정의(§3)보다 앞에 둔다.

In [3]:
PREREG = {
    "notebook": "28_domain_lexicon_v1",
    "builds": "data/scent_knowledge/domain_lexicon_v1.csv",
    "llm_calls": 0,

    # spec.md §4.3 매핑 규칙 (DECISIONS.md N2)
    "rule_min_core_candidates": 2,      # 한 갈래의 core 후보는 2개 이상
    "rule_min_search_results": 3,       # 검색 결과가 3개 미만이면 항목을 넣지 않는다
    "rule_tie_report_only": True,       # 동점 규모는 보고만 하고 게이트로 쓰지 않는다
    "TOP_K": 5,

    # 근거 등급 = 매핑의 출처
    "evidence_tiers": {
        "VERIFIED": "스키마 필드 · accord/note 이름 동일성 · 코퍼스 측정값이 매핑을 결정",
        "TEAM": "팀 문서 perfume_14families_korean_descriptors.md 에 출처 행이 있다",
        "LLM": "출처 없이 AI 초안. 사람 검토 후 TEAM 으로 승격",
    },
    # 매핑 행의 초기 status. 의미 판정을 받지 않았으므로 active 가 아니다.
    "mapping_status": "candidate",
    # FIELD / NO_MAPPING 행은 사실이므로 active
    "factual_status": "active",

    # candidate_type=FIELD 의 허용 값. SCHEMA.md 에 실제로 있는 필드만.
    "allowed_fields": [
        "notes", "notes.tiered.top", "notes.tiered.middle", "notes.tiered.base",
        "longevity", "sillage",
    ],
    "field_source": "SCHEMA.md (perfumes.jsonl dump)",

    "excluded_this_version": {
        "질감층": "spec.md §8 남은 작업 5번 미결정 (무거운·가벼운·강한·묵직한)",
        "은어·비유": "DECISIONS.md N1 — 실사용 1.3%, 후순위",
        "외래어 파생 direct": "spec.md §4.3 — 나열하지 않는다. 25_stage1_scoring_alias_v1.csv 에 이미 있다",
        "부향률": "SCHEMA.md dump 에 concentration 필드가 없다 → NO_MAPPING 으로만 기록",
    },
}

TOP_K = PREREG["TOP_K"]
MIN_CORE = PREREG["rule_min_core_candidates"]
MIN_RESULTS = PREREG["rule_min_search_results"]
ALLOWED_FIELDS = set(PREREG["allowed_fields"])

display(Markdown("```json\n" + json.dumps(PREREG, ensure_ascii=False, indent=2) + "\n```"))

```json
{
  "notebook": "28_domain_lexicon_v1",
  "builds": "data/scent_knowledge/domain_lexicon_v1.csv",
  "llm_calls": 0,
  "rule_min_core_candidates": 2,
  "rule_min_search_results": 3,
  "rule_tie_report_only": true,
  "TOP_K": 5,
  "evidence_tiers": {
    "VERIFIED": "스키마 필드 · accord/note 이름 동일성 · 코퍼스 측정값이 매핑을 결정",
    "TEAM": "팀 문서 perfume_14families_korean_descriptors.md 에 출처 행이 있다",
    "LLM": "출처 없이 AI 초안. 사람 검토 후 TEAM 으로 승격"
  },
  "mapping_status": "candidate",
  "factual_status": "active",
  "allowed_fields": [
    "notes",
    "notes.tiered.top",
    "notes.tiered.middle",
    "notes.tiered.base",
    "longevity",
    "sillage"
  ],
  "field_source": "SCHEMA.md (perfumes.jsonl dump)",
  "excluded_this_version": {
    "질감층": "spec.md §8 남은 작업 5번 미결정 (무거운·가벼운·강한·묵직한)",
    "은어·비유": "DECISIONS.md N1 — 실사용 1.3%, 후순위",
    "외래어 파생 direct": "spec.md §4.3 — 나열하지 않는다. 25_stage1_scoring_alias_v1.csv 에 이미 있다",
    "부향률": "SCHEMA.md dump 에 concentration 필드가 없다 → NO_MAPPING 으로만 기록"
  }
}
```

## 3. 항목 정의

한 표현이 후보를 여러 개 가지면 여러 줄이 된다(`spec.md` §4.3). 문맥 분기가 있으면
갈래마다 `match_condition`이 다르다.

`rationale`은 **사용자 설명에 그대로 쓸 수 있는 문장**으로 적는다(§4.3).
`TEAM` 등급 행은 팀 문서의 어느 절에서 왔는지를 문장에 포함한다.

In [4]:
# (entry_id, expression, aliases, expression_type, target_field, standardness,
#  [ (candidate_type, candidate_name, rank, required, match_condition,
#     evidence_tier, rationale), ... ])
ENTRIES = [

    # ───────────────────────── 우선순위 1. 향수 전문 용어 (실사용 18.1%)
    ("kr.term.top_note", "탑노트", "탑노트|탑|톱노트|탑 노트", "PERFORMANCE",
     "STAGE1_DIRECT", "loanword", [
        ("FIELD", "notes.tiered.top", 1, "core", "", "VERIFIED",
         "탑노트는 향수를 뿌린 직후 먼저 올라오는 향이다. Fragrantica 스키마의 "
         "notes.tiered.top 에 직접 대응한다."),
     ]),
    ("kr.term.middle_note", "미들노트", "미들노트|미들|미들 노트|하트노트|미드노트",
     "PERFORMANCE", "STAGE1_DIRECT", "loanword", [
        ("FIELD", "notes.tiered.middle", 1, "core", "", "VERIFIED",
         "미들노트는 탑노트가 날아간 뒤 중심이 되는 향이다. Fragrantica 스키마의 "
         "notes.tiered.middle 에 직접 대응한다."),
     ]),
    ("kr.term.base_note", "베이스노트", "베이스노트|베이스|베이스 노트|라스트노트",
     "PERFORMANCE", "STAGE1_DIRECT", "loanword", [
        ("FIELD", "notes.tiered.base", 1, "core", "", "VERIFIED",
         "베이스노트는 가장 오래 남는 향이다. Fragrantica 스키마의 "
         "notes.tiered.base 에 직접 대응한다."),
     ]),
    ("kr.term.note", "노트", "노트|향조", "PERFORMANCE", "STAGE1_DIRECT", "loanword", [
        ("FIELD", "notes", 1, "core", "", "VERIFIED",
         "노트는 향수에 들어간 향료를 뜻한다. 어느 단계인지 말하지 않았으므로 "
         "탑·미들·베이스를 구분하지 않고 전체를 가리킨다."),
     ]),
    ("kr.term.longevity", "지속력", "지속력|지속|지속성|지속 시간",
     "PERFORMANCE", "STAGE1_DIRECT", "standard", [
        ("FIELD", "longevity", 1, "core", "", "VERIFIED",
         "지속력은 향이 얼마나 오래 남는지다. Fragrantica 스키마의 longevity "
         "(1~5, 1이 약함)에 대응한다. ⚠ 현재 ERD 에는 이 값을 담을 컬럼이 확인되지 않았다."),
     ]),
    ("kr.term.sillage", "발향", "발향|확산력|실라지|실라주|퍼짐",
     "PERFORMANCE", "STAGE1_DIRECT", "standard", [
        ("FIELD", "sillage", 1, "core", "", "VERIFIED",
         "발향은 향이 주변으로 얼마나 퍼지는지다. Fragrantica 스키마의 sillage "
         "(1~4, 1이 밀착)에 대응한다. ⚠ 현재 ERD 에는 이 값을 담을 컬럼이 확인되지 않았다."),
     ]),
    # 잔향 — 두 뜻이 겹친다. 문맥으로 갈린다.
    ("kr.term.residual", "잔향", "잔향|잔향이|남는 향|끝향",
     "PERFORMANCE", "STAGE1_DIRECT", "standard", [
        ("FIELD", "longevity", 1, "core", "query_contains:오래,길게,긴,남는,좋은,강한",
         "VERIFIED",
         "잔향을 '오래 남는다'는 뜻으로 쓸 때는 지속력을 말한다. Fragrantica 스키마의 "
         "longevity 에 대응한다."),
        ("FIELD", "notes.tiered.base", 2, "core", "", "VERIFIED",
         "잔향을 '어떤 향이 남는가'로 쓸 때는 베이스노트를 말한다. 기본 갈래로 둔다."),
     ]),
    # 대응 필드가 없는 전문 용어
    ("kr.term.concentration", "부향률", "부향률|오드퍼퓸|오드뚜왈렛|EDP|EDT",
     "PERFORMANCE", "NO_MAPPING", "standard", [
        ("", "", 1, "", "", "VERIFIED",
         "부향률은 향료 농도다. SCHEMA.md 를 확인한 결과 dump 에 concentration 필드가 "
         "없어 검색 조건으로 쓸 수 없다. spec.md §6 의 미확인 항목과 같은 내용이다."),
     ]),
    ("kr.term.linear", "리니어", "리니어|리니어한|선형", "PERFORMANCE", "NO_MAPPING",
     "loanword", [
        ("", "", 1, "", "", "VERIFIED",
         "리니어는 시간이 지나도 향이 거의 바뀌지 않는 전개 방식을 뜻한다. "
         "dump 에 향 전개를 담은 필드가 없어 검색 조건으로 쓸 수 없다."),
     ]),

    # ───────────────────────── 우선순위 3. 한국어 드리프트
    ("kr.drift.musk", "머스크", "머스크|머스키|머스크향|화이트머스크|화이트 머스크",
     "DIRECT_SCENT", "STAGE2_BRIDGE", "loanword", [
        ("ACCORD", "musky", 1, "core", "", "VERIFIED",
         "한국어 '머스크'는 대개 화이트머스크, 즉 비누·세탁 느낌을 가리킨다. "
         "Fragrantica 의 musky 는 애니멀릭까지 포괄하므로 범위가 다르다. "
         "musky 를 단독으로 쓰면 코퍼스의 31.2% 가 걸리고 순위가 나오지 않으므로 "
         "반드시 soapy 와 함께 쓴다."),
        ("ACCORD", "soapy", 2, "core", "", "TEAM",
         "팀 문서의 청결감 축이 '비누 같은 / 세탁한 듯한 / 스킨 같은' 을 한 묶음으로 "
         "정리하고 있다. 한국어 화자가 머스크로 뜻하는 쪽이 이 묶음이다."),
        ("ACCORD", "fresh", 3, "optional", "", "LLM",
         "비누·세탁 인상을 더 좁히기 위한 보조 후보다. 출처 없이 고른 것이므로 "
         "사람 검토가 필요하다."),
     ]),

    # ───────────────────────── 우선순위 4. 문맥 분기
    ("kr.ctx.clean", "깨끗한", "깨끗한|깨끗하고|깨끗함|깔끔한|정갈한|단정한",
     "SENSORY", "STAGE2_BRIDGE", "standard", [
        ("ACCORD", "soapy", 1, "core", "query_contains:비누,세탁,빨래,샤워,침구,이불,스킨,뽀송",
         "TEAM",
         "팀 문서의 청결감 축이 '깨끗한' 을 '비누 같은 / 세탁한 듯한 / 샤워한 듯한' 과 "
         "같은 묶음에 두고 있다. 쿼리에 비누·세탁 문맥이 있으면 이 갈래다."),
        ("ACCORD", "fresh", 2, "core", "query_contains:비누,세탁,빨래,샤워,침구,이불,스킨,뽀송",
         "LLM",
         "soapy 를 단독으로 쓸 수 없으므로 짝이 되는 후보다. 출처 없이 고른 것이다."),
        ("ACCORD", "aquatic", 1, "core", "query_contains:물,바다,여름,공기,수영,워터,시원",
         "TEAM",
         "팀 문서의 Water 계열에 '깨끗한 워터 계열' 절이 따로 있다. 쿼리에 물·바다·여름 "
         "문맥이 있으면 이 갈래다."),
        ("ACCORD", "fresh", 2, "core", "query_contains:물,바다,여름,공기,수영,워터,시원",
         "TEAM",
         "같은 절이 맑고 시원한 물 인상을 함께 묶고 있다."),
     ]),
    ("kr.scene.laundry", "빨래", "빨래|빨래향|세탁|세탁한|햇빛에 말린|갓 세탁한",
     "SOURCE", "STAGE2_BRIDGE", "standard", [
        ("ACCORD", "soapy", 1, "core", "", "TEAM",
         "팀 문서의 생활 비유에 '햇빛에 바싹 말린 빨래' 가 있고 청결감 축에 "
         "'세탁한 듯한' 이 있다. 비누 계열이 이 인상의 중심이다."),
        ("ACCORD", "fresh", 2, "core", "", "TEAM",
         "같은 축이 '뽀송한' 을 함께 두고 있다. 마른 빨래의 가볍고 개운한 쪽이다."),
     ]),
    ("kr.sens.cozy", "포근한", "포근한|포근하고|포근함|폭닥한|폭신한|보드라운",
     "SENSORY", "STAGE2_BRIDGE", "standard", [
        ("ACCORD", "powdery", 1, "core", "", "TEAM",
         "포근하다는 촉감 형용사가 후각으로 전이된 표현이다. 팀 문서의 질감 축이 "
         "'파우더리한 / 폭닥한' 을 같은 묶음에 두고 있고, 파우더리한 질감이 그 인상을 만든다."),
        ("ACCORD", "musky", 2, "core", "", "TEAM",
         "팀 문서가 근거로 든 기사 [S29] 가 '머스크를 머금은 폭닥한 향수' 를 다룬다. "
         "머스크가 파우더리한 인상을 감싸는 쪽이다."),
        ("ACCORD", "vanilla", 3, "optional", "", "TEAM",
         "팀 문서의 온도 축이 '따뜻한 / 포근한' 을 함께 둔다. 바닐라의 단맛이 따뜻한 쪽을 "
         "만든다. powdery + musky 만으로 순위를 매기면 5위 동점이 87개가 되는데, "
         "바닐라를 점수에 더하면 9개로 줄어든다. 그래서 필수가 아니라 보조로 둔다."),
     ]),
    ("kr.scene.bedding", "이불", "이불|침구|이불 냄새|코튼 이불|침대",
     "SOURCE", "STAGE2_BRIDGE", "standard", [
        ("ACCORD", "powdery", 1, "core", "", "TEAM",
         "팀 문서의 생활 비유에 '코튼 이불' 과 '깨끗한 침구' 가 있다. 보드라운 천의 "
         "인상이 파우더리한 질감으로 나타난다."),
        ("ACCORD", "soapy", 2, "core", "", "TEAM",
         "'깨끗한 침구' 는 세탁된 상태를 말하므로 청결감 축의 비누 계열이 함께 걸린다."),
        ("ACCORD", "musky", 3, "optional", "", "LLM",
         "감싸는 인상을 더하기 위한 보조 후보다. 출처 없이 고른 것이다."),
     ]),
    ("kr.scene.rainy_forest", "비 오는 숲",
     "비 오는 숲|비 온 뒤 숲|비 온 뒤의 숲|젖은 숲|숲 냄새|숲 속",
     "SCENE", "STAGE2_BRIDGE", "standard", [
        ("ACCORD", "mossy", 1, "core", "", "TEAM",
         "팀 문서의 Mossy Woods 계열에 '숲 이미지 표현' 절이 있고 생활 비유에 "
         "'비 온 뒤 숲' 이 있다. 이끼가 젖은 숲 인상의 중심이다."),
        ("ACCORD", "earthy", 2, "core", "", "TEAM",
         "같은 계열의 습도 표현이 젖은 흙 쪽을 함께 둔다. 비 온 뒤의 땅 냄새다."),
        ("ACCORD", "green", 3, "optional", "", "LLM",
         "잎과 풀의 초록 쪽을 더하는 보조 후보다. 출처 없이 고른 것이다."),
     ]),
    ("kr.sens.cold", "차가운", "차가운|차갑고|차가움|서늘한|쨍한",
     "SENSORY", "STAGE2_BRIDGE", "standard", [
        ("ACCORD", "ozonic", 1, "core", "", "TEAM",
         "팀 문서의 온도 축이 '차가운 / 서늘한' 을 같은 묶음에 두고, Water 계열에 "
         "'오존/메탈릭 워터' 절이 따로 있다. 찬 공기 인상이 오조닉이다."),
        ("ACCORD", "fresh", 2, "core", "", "TEAM",
         "같은 절이 맑고 시원한 쪽을 함께 묶는다."),
     ]),
    ("kr.sens.dewy", "촉촉한", "촉촉한|촉촉하고|촉촉함|물기 어린|물기",
     "SENSORY", "STAGE2_BRIDGE", "standard", [
        ("ACCORD", "aquatic", 1, "core", "", "TEAM",
         "팀 문서의 습도 축이 '촉촉한 / 물기 어린' 을 같은 묶음에 둔다. 물기가 있는 "
         "인상이 아쿠아틱이다."),
        ("ACCORD", "green", 2, "core", "", "TEAM",
         "같은 문서의 Green 계열에 '오이/워터리 그린' 절이 있다. 촉촉한 초록 쪽이다."),
     ]),
    ("kr.scene.resort", "휴양지", "휴양지|리조트|해변|바닷가|여행지|남태평양",
     "SCENE", "STAGE2_BRIDGE", "standard", [
        ("ACCORD", "tropical", 1, "core", "", "TEAM",
         "팀 문서의 생활 비유에 '휴양지의 저녁' 이 있다. 열대 과일과 꽃의 인상이 "
         "트로피컬이다."),
        ("ACCORD", "coconut", 2, "core", "", "LLM",
         "선탠오일과 코코넛의 휴양지 연상을 담은 보조 후보다. 출처 없이 고른 것이므로 "
         "사람 검토가 필요하다."),
     ]),
    ("kr.scene.hotel", "호텔", "호텔|호텔 어메니티|호텔 냄새|로비|어메니티",
     "SCENE", "STAGE2_BRIDGE", "standard", [
        ("ACCORD", "soapy", 1, "core", "", "TEAM",
         "팀 문서의 생활 비유에 '호텔 어메니티' 가 있다. 비누와 샴푸의 인상이 중심이다."),
        ("ACCORD", "white floral", 2, "core", "", "LLM",
         "호텔 어메니티에 흔한 흰 꽃 계열을 담은 보조 후보다. 출처 없이 고른 것이다."),
     ]),

    # ───────────────────────── 우선순위 6. 방언 (DECISIONS.md N1 — 방언 등장의 94.7%)
    ("kr.dial.sweet", "달달", "달달|달달한|달큰한|달콤한|달콤하고",
     "SENSORY", "STAGE2_BRIDGE", "dialect", [
        ("ACCORD", "sweet", 1, "core", "", "VERIFIED",
         "달달하다는 미각 형용사가 후각으로 전이된 표현이며 sweet accord 이름과 뜻이 같다. "
         "다만 sweet 단독으로는 코퍼스가 너무 넓어 순위가 나오지 않는다."),
        ("ACCORD", "caramel", 2, "core", "", "LLM",
         "단맛을 좁히기 위한 짝이다. vanilla 를 쓰면 결과가 너무 넓어 caramel 을 골랐다. "
         "출처 없이 고른 것이므로 사람 검토가 필요하다."),
     ]),

    # ───────────────────────── 우선순위 5. 매핑 금지
    ("kr.nomap.fragrant", "향긋한", "향긋한|향긋하고|향긋함|향긋",
     "SENSORY", "NO_MAPPING", "standard", [
        ("", "", 1, "", "", "VERIFIED",
         "표준국어대사전 정의가 '은근히 향기로운 느낌이 있다' 로 묘사 내용이 없는 순수 "
         "평가어다. 어떤 향인지 지정하지 않으므로 accord 를 붙이지 않는다. "
         "DECISIONS.md N1 의 결정과 같다."),
     ]),
    ("kr.nomap.plain", "무난한", "무난한|무난하고|데일리한|부담없는|튀지 않는",
     "OTHER", "NO_MAPPING", "standard", [
        ("", "", 1, "", "", "VERIFIED",
         "팀 문서가 이 표현에 대해 '14개 패밀리 어디에도 없던 표현인데 질문 글 최빈어' 이며 "
         "'(매핑 근거 미확보)' 라고 직접 적어두었다. 문서의 판단을 그대로 따른다."),
     ]),
    ("kr.nomap.luxurious", "고급스러운", "고급스러운|고급진|우아한|세련된",
     "IMAGE", "NO_MAPPING", "standard", [
        ("", "", 1, "", "", "TEAM",
         "팀 문서에 7번 나오는데 Soft Floral·Amber 등 서로 다른 계열에 흩어져 있다. "
         "특정 향 방향을 지정하지 않는 이미지어이므로 accord 를 붙이지 않는다."),
     ]),
    ("kr.nomap.urban", "도시적인", "도시적인|도시적|모던한",
     "IMAGE", "NO_MAPPING", "standard", [
        ("", "", 1, "", "", "TEAM",
         "팀 문서의 성숙도·이미지 축에 속한 표현이다. 향의 내용이 아니라 인상을 말하므로 "
         "accord 를 붙이지 않는다."),
     ]),
    ("kr.nomap.sexy", "섹시한", "섹시한|관능적인|퇴폐적인",
     "IMAGE", "NO_MAPPING", "standard", [
        ("", "", 1, "", "", "TEAM",
         "팀 문서에서 Amber·Woody Amber 계열 형용사로 나오지만, 파일럿 계획 §11 이 "
         "결과가 stereotype 으로 수렴하는 것을 중단 기준으로 두고 있다. "
         "사람 판정을 받기 전에는 매핑하지 않는다."),
     ]),
    ("kr.nomap.natural", "자연스러운", "자연스러운|자연스럽게|내추럴한",
     "SENSORY", "NO_MAPPING", "standard", [
        ("", "", 1, "", "", "TEAM",
         "팀 문서에서 두 역할로 쓰인다. 자연성 축에서는 '인공적인' 의 반대말이고, "
         "Woods 계열에서는 목재 형용사다. 두 역할이 달라 단일 방향을 지정할 수 없다. "
         "문맥 분기 규칙을 정한 뒤 재검토한다."),
     ]),
]

n_expr = len(ENTRIES)
n_rows = sum(len(e[6]) for e in ENTRIES)
print(f"표현 {n_expr}개 / 행 {n_rows}개")

표현 26개 / 행 44개


## 4. 검증 1 — `candidate_name`이 마스터 목록에 있는가

`spec.md` §4.3: *"기동 시 모든 accord/note 이름이 `accords`/`notes` 테이블에 존재하는지
검사하고, 하나라도 없으면 에러를 내고 기동을 중단한다. FK와 같은 효과다."*

같은 검사를 사전을 만드는 시점에 한다. `FIELD` 는 사전 등록한 허용 목록과 대조한다.

In [5]:
accord_master = pd.read_csv(INPUT_PATHS["accord_dictionary"])
note_master = pd.read_csv(INPUT_PATHS["note_vocabulary"])
ACCORD_SUPPORT = dict(zip(accord_master.accord, accord_master.perfume_count))
NOTE_SUPPORT = dict(zip(note_master.note, note_master.perfume_count))

problems = []
for entry_id, expr, _, _, target_field, _, cands in ENTRIES:
    for ctype, cname, rank, *_ in cands:
        if ctype == "ACCORD" and cname not in ACCORD_SUPPORT:
            problems.append(f"{entry_id} rank{rank}: accord '{cname}' 이 92개 목록에 없다")
        elif ctype == "NOTE" and cname not in NOTE_SUPPORT:
            problems.append(f"{entry_id} rank{rank}: note '{cname}' 이 목록에 없다")
        elif ctype == "FIELD" and cname not in ALLOWED_FIELDS:
            problems.append(f"{entry_id} rank{rank}: field '{cname}' 이 허용 목록에 없다")
        elif ctype == "" and target_field != "NO_MAPPING":
            problems.append(f"{entry_id}: candidate_type 이 비었는데 NO_MAPPING 이 아니다")

if problems:
    for p in problems:
        print(" ", p)
    raise RuntimeError(f"검증 1 실패 — {len(problems)}건")
print(f"검증 1 통과 — candidate_name 전부 마스터 목록에 존재")
print(f"  accord 마스터 {len(ACCORD_SUPPORT)}개 / note 마스터 {len(NOTE_SUPPORT):,}개 "
      f"/ 허용 field {len(ALLOWED_FIELDS)}개")

검증 1 통과 — candidate_name 전부 마스터 목록에 존재
  accord 마스터 92개 / note 마스터 2,523개 / 허용 field 6개


## 5. 검증 2 — 매핑 규칙 (`DECISIONS.md` N2)

두 규칙을 코드가 검사한다.

1. **한 갈래의 `core` 후보는 2개 이상** — 단독 매핑 금지
2. **그 조합으로 검색하면 3개 이상 나온다** — 검색 불가 항목은 넣지 않는다

`5위 동점` 규모는 함께 계산해 보고하지만 게이트로 쓰지 않는다(사전 등록 참조).

**`core` 와 `optional` 을 다르게 다룬다.** `spec.md` §3 이 *"`required: true` 는 모두
만족해야 하고 `required: false` 는 점수에만 기여한다"* 로 정의하므로,
**필터는 `core` 로, 점수는 `core` + `optional` 로** 계산한다.

이 구분이 결과를 바꾼다. `포근한` 의 `core` 는 `powdery` + `musky` 이고 후보가 29,198개인데,
`core` 만으로 순위를 매기면 5위 동점이 **87개**다. `optional` 인 `vanilla` 를 점수에 더하면
**9개**로 줄어든다. 후보를 좁히지 않고 순위만 정리하는 것이 `optional` 의 역할이다.

노트북 27의 계산에 이 구분을 더한 것이다.

In [6]:
perfume_df = pd.read_csv(INPUT_PATHS["perfumes"], usecols=["accords"])
PERFUME_ACCORDS = []
for raw in perfume_df.accords.fillna(""):
    if not raw:
        continue
    parsed = {}
    for part in raw.split("|"):
        if not part:
            continue
        name, sep, strength = part.rpartition(":")
        if sep and strength.isdigit():
            parsed[name] = int(strength)
    if parsed:
        PERFUME_ACCORDS.append(parsed)

# 재현 게이트 — 노트북 27과 같은 대조
gate = [a for a in ACCORD_SUPPORT
        if sum(1 for r in PERFUME_ACCORDS if a in r) != ACCORD_SUPPORT[a]]
if gate:
    raise RuntimeError(f"재현 게이트 실패 — accord {len(gate)}개 불일치")
print(f"재현 게이트 통과 — accord {len(ACCORD_SUPPORT)}개 perfume_count 오차 0 "
      f"(향수 {len(PERFUME_ACCORDS):,}개)")


def search_probe(core, optional=(), k=TOP_K):
    """accord 조합의 후보 수와 상위 k위 진입 동점 규모. dict.

    spec.md §3 을 따른다 — `required: true`(core)는 모두 만족해야 하므로 **필터**로 쓰고,
    `required: false`(optional)는 점수에만 기여하므로 **점수**에만 넣는다.
    optional 을 필터에 넣으면 후보가 실제보다 좁게 나오고, 점수에서 빼면
    동점 규모가 실제보다 크게 나온다.
    """
    core = list(core)
    if not core:
        return {"후보": None, "5위 동점": None}
    score_on = core + list(optional)
    scores = [sum(row.get(t, 0) for t in score_on)
              for row in PERFUME_ACCORDS if all(t in row for t in core)]
    n = len(scores)
    if n == 0:
        return {"후보": 0, "5위 동점": 0}
    if n <= k:
        return {"후보": n, "5위 동점": n}
    cut = sorted(scores, reverse=True)[k - 1]
    return {"후보": n, "5위 동점": sum(1 for s in scores if s >= cut)}


# 갈래별로 core accord 를 모아 규칙을 검사한다
branch_rows = []
violations = []
for entry_id, expr, _, _, target_field, _, cands in ENTRIES:
    if target_field == "NO_MAPPING":
        continue
    branches = {}
    for ctype, cname, rank, required, cond, tier, _ in cands:
        branches.setdefault(cond, []).append((ctype, cname, required))
    for cond, members in branches.items():
        types = {t for t, _, _ in members}
        cores = [n for t, n, r in members if r == "core"]
        accords = [n for t, n, r in members if t == "ACCORD"]
        if types == {"FIELD"}:
            branch_rows.append({"entry_id": entry_id, "표현": expr,
                                "갈래": cond or "(기본)", "종류": "FIELD",
                                "core 수": len(cores), "후보": None, "5위 동점": None,
                                "규칙": "해당 없음"})
            continue
        if len(cores) < MIN_CORE:
            violations.append(f"{entry_id} 갈래 '{cond or chr(40)+chr(41)}': "
                              f"core {len(cores)}개 (규칙 {MIN_CORE}개 이상)")
        probe = search_probe(
            [n for t, n, r in members if t == "ACCORD" and r == "core"],
            [n for t, n, r in members if t == "ACCORD" and r == "optional"])
        if (probe["후보"] or 0) < MIN_RESULTS:
            violations.append(f"{entry_id} 갈래 '{cond or chr(40)+chr(41)}': "
                              f"검색 결과 {probe['후보']}개 (규칙 {MIN_RESULTS}개 이상)")
        branch_rows.append({"entry_id": entry_id, "표현": expr,
                            "갈래": cond or "(기본)",
                            "종류": "+".join(accords),
                            "core 수": len(cores),
                            "후보": probe["후보"], "5위 동점": probe["5위 동점"],
                            "규칙": "통과"})

branch_df = pd.DataFrame(branch_rows)
if violations:
    for v in violations:
        print(" ", v)
    raise RuntimeError(f"검증 2 실패 — {len(violations)}건")
print(f"검증 2 통과 — 갈래 {len(branch_df)}개 전부 규칙 충족\n")
display(branch_df)

재현 게이트 통과 — accord 92개 perfume_count 오차 0 (향수 129,161개)


검증 2 통과 — 갈래 20개 전부 규칙 충족



,entry_id,표현,갈래,종류,core 수,후보,5위 동점,규칙
0,kr.term.top_note,탑노트,(기본),FIELD,1,NaN,NaN,해당 없음
1,kr.term.middle_note,미들노트,(기본),FIELD,1,NaN,NaN,해당 없음
2,kr.term.base_note,베이스노트,(기본),FIELD,1,NaN,NaN,해당 없음
3,kr.term.note,노트,(기본),FIELD,1,NaN,NaN,해당 없음
4,kr.term.longevity,지속력,(기본),FIELD,1,NaN,NaN,해당 없음
5,kr.term.sillage,발향,(기본),FIELD,1,NaN,NaN,해당 없음
6,kr.term.residual,잔향,"query_contains:오래,길게,긴,남는,좋은,강한",FIELD,1,NaN,NaN,해당 없음
7,kr.term.residual,잔향,(기본),FIELD,1,NaN,NaN,해당 없음
8,kr.drift.musk,머스크,(기본),musky+soapy+fresh,2,400.0,5.0,통과
9,kr.ctx.clean,깨끗한,"query_contains:비누,세탁,빨래,샤워,침구,이불,스킨,뽀송",soapy+fresh,2,1049.0,6.0,통과


## 6. 검증 3 — leakage 추적 (`source_query_ids`)

`spec.md` §7.3: *"`korean_scent_lexicon_v0_1.csv`는 golden set 200개에서 추출됐고
`query_ids`에 출처가 기록돼 있다. 같은 200개로 평가하면 암기를 측정한다.
→ `source_query_ids`로 seen/unseen을 나눠 따로 보고한다."*

각 표현이 golden set의 어느 쿼리에서 나왔는지 채운다. 평가할 때 이 컬럼으로 분리한다.

In [7]:
kl = pd.read_csv(INPUT_PATHS["korean_lexicon"], keep_default_na=False, dtype=str)
EXPR_TO_QIDS = dict(zip(kl.expression, kl.query_ids))


def lookup_query_ids(expression, aliases):
    """표현과 별칭이 golden set 쿼리에서 나왔는지 찾는다. str (| 구분)."""
    found = []
    for key in [expression] + aliases.split("|"):
        for qid in EXPR_TO_QIDS.get(key, "").split("|"):
            if qid and qid not in found:
                found.append(qid)
    return "|".join(sorted(found))


seen = 0
for entry_id, expr, aliases, *_ in ENTRIES:
    qids = lookup_query_ids(expr, aliases)
    if qids:
        seen += 1
        print(f"  {expr:12s} golden set {len(qids.split('|')):>2d}개 쿼리에서 등장")
print(f"\n검증 3 — 표현 {len(ENTRIES)}개 중 {seen}개가 golden set 출처를 가진다 "
      f"(평가 시 seen/unseen 분리 필요)")

  머스크          golden set  1개 쿼리에서 등장
  깨끗한          golden set  7개 쿼리에서 등장
  포근한          golden set  1개 쿼리에서 등장
  비 오는 숲       golden set  1개 쿼리에서 등장
  차가운          golden set  1개 쿼리에서 등장
  달달           golden set  2개 쿼리에서 등장
  고급스러운        golden set  2개 쿼리에서 등장
  섹시한          golden set  1개 쿼리에서 등장
  자연스러운        golden set  1개 쿼리에서 등장

검증 3 — 표현 26개 중 9개가 golden set 출처를 가진다 (평가 시 seen/unseen 분리 필요)


## 7. 사전 조립

`spec.md` §4.3의 컬럼 순서를 그대로 따른다. `corpus_support`는 스크립트로 채운다.

In [8]:
COLUMNS = ["entry_id", "expression", "aliases", "expression_type", "target_field",
           "candidate_type", "candidate_name", "rank", "required", "match_condition",
           "rationale", "evidence_tier", "corpus_support", "standardness", "status",
           "source_query_ids"]

rows = []
for entry_id, expr, aliases, etype, target_field, standardness, cands in ENTRIES:
    qids = lookup_query_ids(expr, aliases)
    for ctype, cname, rank, required, cond, tier, rationale in cands:
        if ctype == "ACCORD":
            support = ACCORD_SUPPORT[cname]
        elif ctype == "NOTE":
            support = NOTE_SUPPORT[cname]
        else:
            support = ""
        status = (PREREG["factual_status"]
                  if target_field in ("NO_MAPPING", "STAGE1_DIRECT")
                  else PREREG["mapping_status"])
        rows.append({
            "entry_id": entry_id, "expression": expr, "aliases": aliases,
            "expression_type": etype, "target_field": target_field,
            "candidate_type": ctype, "candidate_name": cname,
            "rank": rank, "required": required, "match_condition": cond,
            "rationale": rationale, "evidence_tier": tier,
            "corpus_support": support, "standardness": standardness,
            "status": status, "source_query_ids": qids,
        })

lexicon_df = pd.DataFrame(rows)[COLUMNS]
print(f"사전 {len(lexicon_df)}행 / 표현 {lexicon_df.expression.nunique()}개\n")
for col in ["target_field", "candidate_type", "evidence_tier", "status",
            "expression_type", "standardness"]:
    print(f"{col:16s} {lexicon_df[col].replace('', '(빈값)').value_counts().to_dict()}")
print()
display(lexicon_df[["entry_id", "expression", "target_field", "candidate_type",
                    "candidate_name", "rank", "required", "evidence_tier",
                    "corpus_support", "status"]])

사전 44행 / 표현 26개

target_field     {'STAGE2_BRIDGE': 28, 'STAGE1_DIRECT': 8, 'NO_MAPPING': 8}
candidate_type   {'ACCORD': 28, 'FIELD': 8, '(빈값)': 8}
evidence_tier    {'TEAM': 23, 'VERIFIED': 14, 'LLM': 7}
status           {'candidate': 28, 'active': 16}
expression_type  {'SENSORY': 15, 'PERFORMANCE': 10, 'SCENE': 7, 'SOURCE': 5, 'DIRECT_SCENT': 3, 'IMAGE': 3, 'OTHER': 1}
standardness     {'standard': 34, 'loanword': 8, 'dialect': 2}



,entry_id,expression,target_field,candidate_type,candidate_name,rank,required,evidence_tier,corpus_support,status
0,kr.term.top_note,탑노트,STAGE1_DIRECT,FIELD,notes.tiered.top,1,core,VERIFIED,,active
1,kr.term.middle_note,미들노트,STAGE1_DIRECT,FIELD,notes.tiered.middle,1,core,VERIFIED,,active
2,kr.term.base_note,베이스노트,STAGE1_DIRECT,FIELD,notes.tiered.base,1,core,VERIFIED,,active
3,kr.term.note,노트,STAGE1_DIRECT,FIELD,notes,1,core,VERIFIED,,active
4,kr.term.longevity,지속력,STAGE1_DIRECT,FIELD,longevity,1,core,VERIFIED,,active
5,kr.term.sillage,발향,STAGE1_DIRECT,FIELD,sillage,1,core,VERIFIED,,active
6,kr.term.residual,잔향,STAGE1_DIRECT,FIELD,longevity,1,core,VERIFIED,,active
7,kr.term.residual,잔향,STAGE1_DIRECT,FIELD,notes.tiered.base,2,core,VERIFIED,,active
8,kr.term.concentration,부향률,NO_MAPPING,,,1,,VERIFIED,,active
9,kr.term.linear,리니어,NO_MAPPING,,,1,,VERIFIED,,active


### 근거 등급별로 무엇이 들어왔는가

`LLM` 등급 행이 사람 검토 대기 목록이다. `spec.md` §4.3의 승격 경로를 따른다.

In [9]:
for tier in ["VERIFIED", "TEAM", "LLM"]:
    sub = lexicon_df[lexicon_df.evidence_tier == tier]
    print(f"\n=== {tier}  {len(sub)}행")
    for r in sub.to_dict("records"):
        target = r["candidate_name"] or "(매핑 없음)"
        print(f"  {r['expression']:12s} → {target:22s} {r['required'] or '-':8s} "
              f"{'' if r['corpus_support'] == '' else format(r['corpus_support'], ',')}")


=== VERIFIED  14행
  탑노트          → notes.tiered.top       core     
  미들노트         → notes.tiered.middle    core     
  베이스노트        → notes.tiered.base      core     
  노트           → notes                  core     
  지속력          → longevity              core     
  발향           → sillage                core     
  잔향           → longevity              core     
  잔향           → notes.tiered.base      core     
  부향률          → (매핑 없음)                -        
  리니어          → (매핑 없음)                -        
  머스크          → musky                  core     41,154
  달달           → sweet                  core     60,625
  향긋한          → (매핑 없음)                -        
  무난한          → (매핑 없음)                -        

=== TEAM  23행
  머스크          → soapy                  core     1,720
  깨끗한          → soapy                  core     1,720
  깨끗한          → aquatic                core     8,872
  깨끗한          → fresh                  core     33,478
  빨래           → soapy           

## 8. 저장

In [10]:
tier_counts = lexicon_df.evidence_tier.value_counts().to_dict()
status_counts = lexicon_df.status.value_counts().to_dict()
llm_rows = lexicon_df[lexicon_df.evidence_tier == "LLM"]
nomap = lexicon_df[lexicon_df.target_field == "NO_MAPPING"]

# 검색 성립성 표에는 accord 갈래만 넣는다. FIELD 는 검색 조건이 아니다.
searchable = branch_df[branch_df["종류"] != "FIELD"]
branch_table = "\n".join(
    f"| {r['표현']} | `{r['종류']}` | {r['갈래']} | "
    f"{int(r['후보']):,} | {int(r['5위 동점']):,} |"
    for r in searchable.to_dict("records"))
llm_table = "\n".join(
    f"| {r['expression']} | `{r['candidate_name']}` | {r['required']} | "
    f"{format(r['corpus_support'], ',') if r['corpus_support'] != '' else ''} |"
    for r in llm_rows.to_dict("records"))
nomap_table = "\n".join(
    f"| {r['expression']} | {r['evidence_tier']} |" for r in nomap.to_dict("records"))

report_text = f"""# Domain Lexicon v1 빌드 리포트

`data/scent_knowledge/domain_lexicon_{LEXICON_VERSION}.csv` 를 만든 기록이다.

## 무엇을 만들었는가

| | |
|---|---:|
| 표현 | {lexicon_df.expression.nunique()} |
| 행 | {len(lexicon_df)} |
| `VERIFIED` | {tier_counts.get('VERIFIED', 0)} |
| `TEAM` | {tier_counts.get('TEAM', 0)} |
| `LLM` (사람 검토 대기) | {tier_counts.get('LLM', 0)} |
| `NO_MAPPING` | {len(nomap)} |
| `status=active` | {status_counts.get('active', 0)} |
| `status=candidate` | {status_counts.get('candidate', 0)} |

`spec.md` §4.3의 목표는 30~50개 항목이었다.

## 스키마 변경 1건

`candidate_type` 에 **`FIELD`** 를 추가했다. 우선순위 1번(향수 전문 용어, 실사용 18.1%)이
`탑노트 → notes.tiered.top` 처럼 Fragrantica 스키마 필드를 가리키는데
기존 `ACCORD` / `NOTE` 로는 담을 칸이 없었다. 사용자 승인 2026-09-10.

허용 필드는 `SCHEMA.md` 에 실제로 있는 것만 넣었다 — {sorted(ALLOWED_FIELDS)}.

**`부향률`은 매핑하지 않았다.** `SCHEMA.md` 를 확인한 결과 dump 에 `concentration` 필드가
없다. `spec.md` §6 의 미확인 항목과 같은 내용이다.

## 근거 등급을 매핑의 출처로 정의했다

| 등급 | 뜻 |
|---|---|
| `VERIFIED` | 스키마 필드 · accord/note 이름 동일성 · 코퍼스 측정값이 매핑을 결정 |
| `TEAM` | 팀 문서 `perfume_14families_korean_descriptors.md` 에 출처 행이 있다 |
| `LLM` | 출처 없이 AI 초안. 사람 검토 후 `TEAM` 으로 승격 |

`AGENTS.md` 의 *"Do not invent mappings such as an abstract phrase to scent features
without a defined evidence or modeling method"* 를 따라, 세 방법 밖의 매핑은 만들지 않고
근거가 없으면 `NO_MAPPING` 으로 뒀다.

## 검증 3건

1. **`candidate_name` 존재** — accord {len(ACCORD_SUPPORT)}개 / note {len(NOTE_SUPPORT):,}개 /
   허용 field {len(ALLOWED_FIELDS)}개 마스터 목록과 대조. 전부 통과
2. **매핑 규칙** (`DECISIONS.md` N2) — 갈래 {len(branch_df)}개 전부
   `core` 2개 이상 + 검색 결과 3개 이상. 전부 통과
3. **leakage 추적** — `source_query_ids` 를 채워 seen/unseen 분리 가능하게 함

재현 게이트로 accord {len(ACCORD_SUPPORT)}개 `perfume_count` 를 오차 0으로 재현한 뒤 계산했다.

## 갈래별 검색 성립성

`core` 로 필터하고 `core` + `optional` 로 점수를 매긴 결과다(`spec.md` §3).
`FIELD` 갈래 {len(branch_df) - len(searchable)}개는 검색 조건이 아니므로 표에서 뺐다.

| 표현 | core accord | 갈래 조건 | 후보 | 5위 동점 |
|---|---|---|---:|---:|
{branch_table}

## 사람 검토 대기 — `LLM` 등급 {len(llm_rows)}행

출처 없이 AI 가 고른 후보다. `spec.md` §4.3 의 승격 경로대로 사람 검토를 거쳐
`TEAM` 으로 올린다.

| 표현 | 후보 | required | 코퍼스 |
|---|---|---|---:|
{llm_table}

## 매핑하지 않은 표현 {len(nomap)}개

| 표현 | 근거 등급 |
|---|---|
{nomap_table}

## 한계

- **`TEAM` 등급은 "팀 문서에 표현이 있다"까지만 보증한다.** 문서는 표현을 향 계열로
  정리한 것이고 accord 를 지정하지 않았다. 계열 → accord 단계는 이 노트북의 판단이다.
  문서 자신의 경고 — *"`포근한`, `깨끗한` 같은 단어는 패밀리를 단독 판별하는 단어가
  아니라 방향을 잡는 단어다."*
- **의미 판정을 받지 않았다.** 검색이 되는지는 계산했지만 `soapy+fresh` 가 `빨래` 의
  옳은 번역인지는 사람이 판정할 문제다. 매핑 행은 전부 `status=candidate` 다
- **질감층을 넣지 않았다.** `spec.md` §8 남은 작업 5번 미결정
- **은어·비유를 넣지 않았다.** `DECISIONS.md` N1 — 실사용 1.3%, 후순위
- **`잔향` 은 두 뜻이 겹친다.** 지속력인지 베이스노트인지 문맥으로 갈리므로 두 행으로 뒀다
- **`지속력`·`발향` 은 dump 필드에 대응하지만 ERD 컬럼이 확인되지 않았다**
- 커버리지를 측정하지 않았다. `spec.md` §4.3 이 인용한 8.2%/15.8% 는 다른 사전의 값이다

## 재현 방법

```bash
cd EDA
export PYTHONIOENCODING=utf-8
# 28_domain_lexicon_v1.ipynb 를 REPORT_ONLY=False 로 실행
# API 호출 0회. 검증 3건과 재현 게이트가 먼저 통과해야 저장된다
```

## 관련 자료

- `docs/spec.md` §4.3 — 사전 스키마와 우선순위
- `docs/DECISIONS.md` N1 — 방언·은어 후순위 / N2 — accord 조합 매핑
- `docs/nlr_engineering_notes.md` 1·2·8번 — 코퍼스 지지도, 머스크 드리프트, 단독 금지
- `27_bridge_accord_search_feasibility.ipynb` — 검색 성립성 계산 방법
- `analysis_outputs/25_stage1_scoring_alias_v1.csv` — 채점기용 별칭표 (복제하지 않음)
- `data/scent_knowledge/source/perfume_14families_korean_descriptors.md` — `TEAM` 등급의 출처
"""

write_output(OUTPUT_PATHS["lexicon"],
             lambda p: lexicon_df.to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["report"],
             lambda p: p.write_text(report_text, encoding="utf-8"))

if not REPORT_ONLY:
    reread = pd.read_csv(OUTPUT_PATHS["lexicon"], keep_default_na=False, dtype=str)
    if len(reread) != len(lexicon_df):
        raise ValueError(f"저장 결과 행 수 불일치: {len(reread)} != {len(lexicon_df)}")
    if list(reread.columns) != COLUMNS:
        raise ValueError("저장 결과 컬럼 불일치")
    print(f"검증 통과 — 사전 {len(reread)}행 / 컬럼 {len(reread.columns)}개")

저장: data\scent_knowledge\domain_lexicon_v1.csv
저장: analysis_outputs\28_domain_lexicon_build_report.md
검증 통과 — 사전 44행 / 컬럼 16개


## 9. 가드 검증

In [11]:
input_hashes_after = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
changed = [k for k in input_hashes_before
           if input_hashes_before[k] != input_hashes_after[k]]
if changed:
    raise RuntimeError(f"입력이 변경됐습니다: {changed}")
print("입력 해시 동일")

for path in sorted(PROTECTED_EXTRA):
    print(f"보호 대상 미변경 확인: {path.name}  {'존재' if path.is_file() else '없음'}")

print("생성한 출력:")
for label, path in OUTPUT_PATHS.items():
    mark = "" if path.is_file() else "  (REPORT_ONLY로 미생성)"
    print(f"  {label}: {path.relative_to(PROJECT_ROOT)}{mark}")

입력 해시 동일


보호 대상 미변경 확인: 16_golden_set_quality_audit_candidates.csv  존재
보호 대상 미변경 확인: 22_pilot_human_evaluation.csv  존재
보호 대상 미변경 확인: 13_stage1_golden_set_v1_200.xlsx  존재
보호 대상 미변경 확인: 16_golden_set_quality_audit_reviewed.csv  존재
생성한 출력:
  lexicon: data\scent_knowledge\domain_lexicon_v1.csv
  report: analysis_outputs\28_domain_lexicon_build_report.md
